In [ ]:
# Imports.
import dv_processing as dv  # dv_processing: event-camera I/O.
import numpy as np  # NumPy.
import os  # OS interface.


def read_aedat4_events(file_path, max_events=None):
    """
    Read the event stream from an AEDAT4 file.
    Args: file_path - AEDAT4 file path.
        max_events - maximum number of events to read; None reads all.
    Returns: events_data - dict of event arrays.
    """
    
    # Open the AEDAT4 recording.
    recording = dv.io.MonoCameraRecording(file_path)
    
    # Check that the event stream is present.
    if not recording.isEventStreamAvailable():
        print("Error: file has no event stream.")
        return None
    
    
    # Initialize event-data buffers.
    timestamps = []  # Timestamps.
    x_coords = []  # x coordinates.
    y_coords = []  # y coordinates.
    polarities = []  # Polarity.
    
    # Time-slice window (100 ms).
    time_slice_us = 100000  # Slice length in microseconds.
    current_time = 0  # Current cursor.
    first_timestamp = None  # Timestamp of the first event.
    
    # Use the first event to anchor the timeline.
    first_events = recording.getNextEventBatch()
    if first_events is not None and len(first_events) > 0:
        first_timestamp = first_events[0].timestamp()
        current_time = first_timestamp
        print(f"First event timestamp: {first_timestamp}")
    
    # Read every event.
    while True:
        # Stop early if a max_events cap was provided.
        if max_events is not None and len(timestamps) >= max_events:
            print(f"Reached the configured event cap: {max_events}")
            break
        
        # Read events within the time slice.
        events = recording.getEventsTimeRange(current_time, current_time + time_slice_us)
        
        # Stop when no more events remain.
        if events is None or len(events) == 0:
            break
        
        # Extract event fields.
        for event in events:
            # Stop early if a max_events cap was provided.
            if max_events is not None and len(timestamps) >= max_events:
                break
                
            timestamps.append(event.timestamp())  # append timestamp
            x_coords.append(event.x())  # append x
            y_coords.append(event.y())  # append y
            polarities.append(event.polarity())  # append polarity
        
        # Advance the cursor.
        current_time += time_slice_us
    
    # Convert to numpy arrays.
    timestamps = np.array(timestamps)
    x_coords = np.array(x_coords)
    y_coords = np.array(y_coords)
    polarities = np.array(polarities)
    
    # Build the events dict.
    events_data = {
        'timestamps': timestamps,
        'x_coords': x_coords,
        'y_coords': y_coords,
        'polarities': polarities
    }
    
    print(f"Read {len(timestamps)} events.")
    return events_data


def read_all_frames(file_path, frame_index=None, output_path="./output"):
    """
    Read all frames from an AEDAT4 file; optionally save a specific frame.
    
    Args:
        file_path: AEDAT4 file path.
        frame_index: index of the frame to save; None saves nothing extra.
        output_path: output directory; defaults to "./output".
    
    Returns:
        frames: list of frames, or None if the file has no frame stream.
    """
    # Create the output directory.
    os.makedirs(output_path, exist_ok=True)
    
    # Open the AEDAT4 recording.
    recording = dv.io.MonoCameraRecording(file_path)
    
    # Check that the frame stream is present.
    if not recording.isFrameStreamAvailable():
        print("Error: file has no frame stream.")
        return None
    
    # Storage for all frames.
    frames = []
    frame_count = 0
    
    # Read every frame.
    while True:
        frame = recording.getNextFrame()
        if frame is None:
            break
        
        # Convert to numpy arrays.
        if hasattr(frame.image, 'numpy'):
            image_array = frame.image.numpy()
        else:
            image_array = np.array(frame.image)
        
        frames.append({
            'index': frame_count,
            'timestamp': frame.timestamp,
            'image': image_array,
            'shape': image_array.shape,
            'dtype': image_array.dtype
        })
        
        frame_count += 1
    
    print(f"Read {frame_count} frames.")
    
    # Save a specific frame.
    if frames and frame_index is not None:
        if 0 <= frame_index < len(frames):
            selected_frame = frames[frame_index]
            import cv2
            
            # Ensure the dtype is correct.
            image_to_save = selected_frame['image']
            if image_to_save.dtype != np.uint8:
                image_to_save = image_to_save.astype(np.uint8)
            
            filename = f"frame_{frame_index:04d}.png"
            file_path_full = os.path.join(output_path, filename)
            cv2.imwrite(file_path_full, image_to_save)
            print(f"Saved frame {frame_index} to {file_path_full}")
            print(f"Frame timestamp: {selected_frame['timestamp']}")
            print(f"Frame shape: {selected_frame['shape']}")
        else:
            print(f"Error: frame index {frame_index} out of range (0-{len(frames)-1}).")
    
    return frames


def play_frames_with_controls(file_path, fps=30):
    """
    Frame playback with keyboard controls.
    
    Args:
        file_path: AEDAT4 file path.
        fps: playback frame rate (default 30 fps).
    """
    import cv2
    import time
    
    # Read every frame.
    recording = dv.io.MonoCameraRecording(file_path)
    if not recording.isFrameStreamAvailable():
        print("Error: file has no frame stream.")
        return
    
    frames = []
    while True:
        frame = recording.getNextFrame()
        if frame is None:
            break
        
        if hasattr(frame.image, 'numpy'):
            image_array = frame.image.numpy()
        else:
            image_array = np.array(frame.image)
        
        if image_array.dtype != np.uint8:
            image_array = image_array.astype(np.uint8)
        
        frames.append(image_array)
    
    if not frames:
        print("No frame data found.")
        return
    
    print(f"Read {len(frames)} frames in total.")
    print("Controls:")
    print("ESC          quit")
    print("Space        pause / resume")
    print("Left arrow   previous frame")
    print("Right arrow  next frame")
    print("+ / -        adjust playback speed")
    
    # Playback state.
    current_frame = 0
    paused = False
    current_fps = fps
    
    cv2.namedWindow("Frame Player", cv2.WINDOW_NORMAL)
    
    while True:
        # Show the current frame.
        frame = frames[current_frame].copy()
        
        # Overlay info text.
        info_text = f"Frame: {current_frame+1}/{len(frames)} | FPS: {current_fps}"
        cv2.putText(frame, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        if paused:
            cv2.putText(frame, "PAUSED", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        cv2.imshow("Frame Player", frame)
        
        # Wait for a key press.
        if paused:
            key = cv2.waitKey(0) & 0xFF
        else:
            key = cv2.waitKey(int(1000 / current_fps)) & 0xFF
        
        # Handle key press.
        if key == 27:  # ESC
            break
        elif key == ord(' '):  # space — pause / resume
            paused = not paused
        elif key == 81:  # left arrow — previous frame
            current_frame = max(0, current_frame - 1)
        elif key == 83:  # right arrow — next frame
            current_frame = min(len(frames) - 1, current_frame + 1)
        elif key == ord('+') or key == ord('='):  # speed up
            current_fps = min(60, current_fps + 5)
        elif key == ord('-'):  # slow down
            current_fps = max(1, current_fps - 5)
        
        # Auto-advance unless paused.
        if not paused:
            current_frame = (current_frame + 1) % len(frames)
    
    cv2.destroyAllWindows()

In [ ]:
def events_to_image(events_data):
    """
    Convert event data to a grayscale image (dynamic resolution + log scaling).
    Args: events_data - dict containing x_coords and y_coords.
    Returns: image - grayscale numpy array.
    """
    # Read coordinates.
    x_coords = events_data['x_coords']
    y_coords = events_data['y_coords']
    
    # Compute dynamic resolution.
    x_min, x_max = np.min(x_coords), np.max(x_coords)
    y_min, y_max = np.min(y_coords), np.max(y_coords)
    
    # Required image dimensions.
    width = x_max - x_min + 1
    height = y_max - y_min + 1
    
    print(f"Coordinate range: X({x_min}-{x_max}), Y({y_min}-{y_max})")
    print(f"Dynamic resolution: {width} x {height}")
    
    # Allocate a blank image with a large dtype to avoid overflow.
    image = np.zeros((height, width), dtype=np.uint32)
    
    # Convert coordinates to offsets from the minimum.
    x_offset = x_coords - x_min
    y_offset = y_coords - y_min
    
    # Accumulate event counts per pixel.
    for x, y in zip(x_offset, y_offset):
        image[y, x] += 1

    # Remove the brightest outlier pixels.
    max_value = np.max(image)
    if max_value > 0:
        print(f"Max event count: {max_value}")
        
        # Locate the max-value pixels and zero them out.
        max_pixels = (image == max_value)
        num_max_pixels = np.sum(max_pixels)
        print(f"Max-value pixel count: {num_max_pixels}")
        
        # Zero out the max-value pixels.
        image[max_pixels] = 0
        
        print(f"Max event count after outlier removal: {np.max(image)}")
    
    # Log-transform and normalize to [0, 255].
    if np.max(image) > 0:
        print(f"Non-zero pixel count: {np.count_nonzero(image)}")
        
        # log1p = log(1 + x); avoids log(0).
        image_log = np.log1p(image)
        
        print(f"After log transform: {np.min(image_log)} - {np.max(image_log)}")
        
        # Normalize to [0, 255].
        image_float = image_log.astype(np.float32)
        
        # Linearly map to [0, 255].
        min_val = np.min(image_float)
        max_val = np.max(image_float)
        
        if max_val > min_val:
            image_float = (image_float - min_val) * 255.0 / (max_val - min_val)
        else:
            image_float = np.zeros_like(image_float)
        
        image = image_float.astype(np.uint8)
        
        print(f"Final pixel range: {np.min(image)} - {np.max(image)}")
    
    return image

def save_event_image(image, output_path="./output", filename="event_image.png"):
    """
    Save the event image to a file.
    
    Args:
        image: image array.
        output_path: output directory; defaults to "./output".
        filename: output filename (default "event_image.png").
    """
    import cv2
    
    # Create the output directory.
    os.makedirs(output_path, exist_ok=True)
    
    # Build the full file path.
    file_path_full = os.path.join(output_path, filename)
    
    cv2.imwrite(file_path_full, image)
    print(f"Image saved to: {file_path_full}")



In [ ]:
# Example usage — replace with your AEDAT4 path.
file_path = "./data/cifar10_xdvs_preview/dv_output_20001_cat_2_9875_20250714_141817.aedat4"  # Replace with your AEDAT4 file path.
output_path = "./aedat_preview"  # Output directory.


# Read event data.
events_data = read_aedat4_events(file_path, max_events=30000)

# Example: convert events to an image.
# Convert event data to a grayscale image.
event_image = events_to_image(events_data)
# Print image info.
print(f"Image shape: {event_image.shape}")
# Save the image to the output directory.
save_event_image(event_image, output_path, "event_image.png")


# Read a specific frame and save it.
read_all_frames(file_path, frame_index=175, output_path=output_path)

# Play all frames (auto-loop; space to pause, Esc to quit).
play_frames_with_controls(file_path, fps=30)
